# Hadoop MapReduce: Student Examination Marks Analysis
Complete Google Colab implementation following Steps 1 to 22.

### Step 1. Check Colab Readiness

In [ ]:
print("Google Colab is ready")

### Step 2. Check the Operating System

In [ ]:
!uname -a

### Step 3. Check Python

In [ ]:
!python3 --version

### Step 4. Install Java 17

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless

### Step 5. Verify Java

In [ ]:
!java -version

### Step 6. Download Hadoop 3.4.1

In [ ]:
!wget -q --show-progress -O /content/hadoop.tar.gz \
https://archive.apache.org/dist/hadoop/common/hadoop-3.4.1/hadoop-3.4.1.tar.gz
!ls -lh /content/hadoop.tar.gz

### Step 7. Extract Hadoop

In [ ]:
!tar -xzf /content/hadoop.tar.gz -C /content
!mv /content/hadoop-3.4.1 /content/hadoop
!ls /content/hadoop

### Step 8. Set Up Hadoop and Java Paths

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop"
os.environ["PATH"] = "/content/hadoop/bin:" + os.environ["PATH"]
os.environ["HADOOP_OPTS"] = "-Djava.library.path=/content/hadoop/lib/native"
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

### Step 9. Check Hadoop Installation

In [ ]:
!hadoop version

### Step 10. Create the Examination Dataset

In [ ]:
data = """StudentID,Name,Subject,Marks
101,Rahul,Math,85
102,Priya,Math,92
103,Amit,Math,35
104,Neha,Math,78
105,Ravi,Math,60
101,Rahul,Physics,75
102,Priya,Physics,88
103,Amit,Physics,42
104,Neha,Physics,95
105,Ravi,Physics,30
101,Rahul,Computer,90
102,Priya,Computer,96
103,Amit,Computer,55
104,Neha,Computer,80
105,Ravi,Computer,35
"""
with open("/content/students.csv", "w") as f:
    f.write(data)
print("Dataset created")

### Step 11. Look at the Dataset

In [ ]:
!cat /content/students.csv

### Step 12. Create the Mapper Program

In [ ]:
%%writefile /content/mapper.py
import csv
import sys

reader = csv.reader(sys.stdin)
for row in reader:
    if not row or row[0] == "StudentID":
        continue
    try:
        student_id, name, subject, marks = row
        marks = int(marks)
        if not 0 <= marks <= 100:
            continue
        print(f"{subject}\t{marks}")
    except (ValueError, IndexError):
        continue

### Step 13. Create the Reducer Program

In [ ]:
%%writefile /content/reducer.py
import sys

def show(subject, marks):
    if not marks:
        return
    count = len(marks)
    avg = sum(marks) / count
    passed = sum(m >= 40 for m in marks)
    failed = count - passed
    print(f"{subject}\t{count}\t{avg:.2f}\t"
          f"{max(marks)}\t{min(marks)}\t{passed}\t{failed}")

current = None
marks_list = []

for line in sys.stdin:
    try:
        subject, value = line.rstrip("\n").split("\t", 1)
        mark = int(value)
    except ValueError:
        continue
    if subject != current:
        if current is not None:
            show(current, marks_list)
        current = subject
        marks_list = []
    marks_list.append(mark)

if current is not None:
    show(current, marks_list)

### Step 14. Test the Mapper Alone

In [ ]:
!python3 /content/mapper.py < /content/students.csv

### Step 15. Test Mapper, Sorting and Reducer

In [ ]:
!python3 /content/mapper.py < /content/students.csv | sort | python3 /content/reducer.py

### Step 16. Find the Hadoop Streaming JAR

In [ ]:
import glob
jars = glob.glob("/content/hadoop/share/hadoop/tools/lib/hadoop-streaming-*.jar")
assert jars, "Hadoop Streaming JAR not found"
streaming_jar = jars[0]
print(streaming_jar)

### Step 17. Create the Hadoop Input Directory

In [ ]:
!mkdir -p /content/marks_input
!cp /content/students.csv /content/marks_input/students.csv
!ls -l /content/marks_input

### Step 18. Run the Real Hadoop Streaming MapReduce Job

In [ ]:
import subprocess
import shutil

shutil.rmtree("/content/marks_output", ignore_errors=True)
cmd = ["hadoop", "jar", streaming_jar,
       "-D", "mapreduce.framework.name=local",
       "-D", "fs.defaultFS=file:///",
       "-files", "/content/mapper.py,/content/reducer.py",
       "-mapper", "python3 mapper.py",
       "-reducer", "python3 reducer.py",
       "-input", "/content/marks_input",
       "-output", "/content/marks_output"]
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout[-2500:])
print(result.stderr[-3500:])
print("Return code:", result.returncode)

### Step 19. Read the Hadoop Output

In [ ]:
!cat /content/marks_output/part-*

### Step 20. Create a Labeled Results Table

In [ ]:
import pandas as pd
import glob

files = glob.glob("/content/marks_output/part-*")
columns = ["Subject", "Students", "Average", "Highest", "Lowest", "Passed", "Failed"]
results = pd.concat([pd.read_csv(f, sep="\t", header=None, names=columns) for f in files])
display(results.sort_values("Subject"))

### Step 21. Draw a Bar Chart

In [ ]:
import matplotlib.pyplot as plt

results = results.sort_values("Subject")
plt.bar(results["Subject"], results["Average"])
plt.ylim(0, 100)
plt.xlabel("Subject")
plt.ylabel("Average marks")
plt.title("Subject-wise Average Examination Marks")
plt.show()

### Step 22. Download the Output

In [ ]:
results.to_csv("/content/student_marks_results.csv", index=False)
from google.colab import files
files.download("/content/student_marks_results.csv")